[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uahccre/ncs_workshop/blob/main/cnn_training.ipynb)

## Making a Copy of the Notebook
1. Click the "Open in Colab" Button
2. The file will open in Read Only mode. Go to File -> Save a Copy in Drive
3. A new tab will open with your editable copy for the rest of the lab.
4. Go to "Edit" and select "Clear All Outputs" to ensure we're starting fresh.

## Training A CNN on CIFAR-10
We will do a hands on walkthrough that does the following:
- Loads real images
- Create a Convolutional Neural Network (CNN)
- Watch the untrained CNN fail
- Train the CNN
- Test it
- See how much it learned

## Setting up the Runtime
Before we run the notebook, do the following: go to **Runtime** -> **Change runtime type** -> **T4 GPU** -> **Save**.

The training will run much faster on a GPU as opposed to the default CPU.


## Importing the Required Libraries
We'll use `torch` to create, train and run our CNN model.

We'll use `torchvision` to load our CIFAR-10 dataset.

We'll use `matplotlib` to visualize our results.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [ ]:
# Verify our notebook is set up to use the T4 GPU
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", device)

## Downloading the Data
We'll use the UAH GitHub mirror to download our dataset.

In [ ]:
import os, urllib.request
os.makedirs("./data", exist_ok=True)
dest = "./data/cifar-10-python.tar.gz"
if not os.path.exists(dest):
    url = "https://github.com/uahccre/ncs_workshop/releases/download/data/cifar-10-python.tar.gz"
    print("Downloading CIFAR-10 from the workshop mirror…")
    urllib.request.urlretrieve(url, dest)
    print("Done.")
else:
    print("CIFAR-10 archive already present.")

In [ ]:
# Load the CIFAR-10 data set. It contains 32x32 color images from 10 everyday classes
transform = transforms.ToTensor()
train_data = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_data = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=128, shuffle=False)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
print(f"{len(train_data):,} training images, {len(test_data):,} test images")

## Why a CNN and not a plain neural network?

A CIFAR-10 image is 32 × 32 × 3 = **3,072 numbers**. If we flattened it and fed it to
an ordinary fully-connected layer with 128 units, that *one* layer would need
3,072 × 128 ≈ **393,000 weights** and it would treat the top-left pixel and the
bottom-right pixel as completely unrelated, throwing away the fact that neighboring
pixels form edges, textures, and shapes.

A **convolutional** layer does the opposite. Our first conv layer uses 32 small 3 × 3
filters that slide across the image, so it needs only 32 × (3 × 3 × 3 + 1) ≈ **900
weights** — roughly **400× fewer** and because each filter looks at a local
neighborhood, spatial structure is preserved.

Where that filter count comes from, for one filter:
- **3 × 3** — the filter's height × width (a small sliding window)
- **× 3** — one value per color channel (red, green, blue)
- **+ 1** — a single bias term added to the result

That's 27 + 1 = **28 numbers per filter**, and 32 filters gives 32 × 28 = **896**.
That's the core idea of a CNN: far fewer parameters, and a built-in assumption that
*location matters*.

## Peek at the Data

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].permute(1, 2, 0)) # C,H,W -> H,W,C for display
    ax.set_title(class_names[labels[i].item()], fontsize=11)
    ax.axis("off")
plt.suptitle("Sample CIFAR-10 Images", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Define the CNN Model

In [ ]:
class TinyCNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(3, 32, kernel_size=3) # 3 color channels in, 32 filters
    self.pool = nn.MaxPool2d(2, 2)
    self.conv2 = nn.Conv2d(32, 64, kernel_size=3) # 32 in from Conv1, 64 filters
    self.fc1 = nn.Linear(64 * 6 * 6, 128) # 64 maps of 6x6 after two Conv+Pool, 128 features out
    self.fc2 = nn.Linear(128, 10) # 10 class scores

  def forward(self, x):
    # This is the actual prediction method the network uses for inference
    x = self.pool(torch.relu(self.conv1(x))) # 32 -> 30 -> 15
    x = self.pool(torch.relu(self.conv2(x))) # 15 -> 13 -> 6
    x = torch.flatten(x, 1) # Create a 1D vector from the pooling outputs
    x = torch.relu(self.fc1(x))
    return self.fc2(x)

model = TinyCNN().to(device)
print(f"Model ready: {sum(p.numel() for p in model.parameters()):,} parameters")


### What each layer is doing

Reading the model from input to output:

- **`conv1` / `conv2` (Conv2d):** the pattern detectors. Each applies a stack of
  sliding 3×3 filters. Early filters pick up simple things (edges, color blobs)
  and the later layer combines those into more complex shapes. `conv1` produces 32
  feature maps, `conv2` produces 64.
- **`relu` (activation):** keeps the positive signal and zeroes out the rest. This
  non-linearity is what lets the network learn patterns more complex than straight
  lines.
- **`pool` (MaxPool2d 2×2):** shrinks each feature map by keeping only the strongest
  value in every 2×2 window. It cuts the data size and makes the network a little
  tolerant to *where* in the image a feature appears. This is why 32×32 becomes 15×15,
  then 6×6.
- **`flatten`:** turns the stack of 2D feature maps (64 maps of 6×6) into one long
  vector of 2,304 numbers per image, so a standard dense layer can read it.
- **`fc1` / `fc2` (Linear):** the decision-makers. They combine the extracted
  features into a final answer. `fc2` outputs 10 numbers, one score per class, and
  the highest score is the prediction.

The pattern **convolve $\rightarrow$ activate $\rightarrow$ pool**, repeated, then **flatten $\rightarrow$ classify**
is the backbone of almost every image CNN, from this tiny one to ResNet.

### Define Helper Functions

In [ ]:
@torch.no_grad()
def test_accuracy(model):
    model.eval()
    correct = total = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return 100 * correct / total

@torch.no_grad()
def show_predictions(model, title):
    model.eval()
    images, labels = next(iter(test_loader))
    preds = model(images.to(device)).argmax(1).cpu()
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i].permute(1, 2, 0))
        right = preds[i].item() == labels[i].item()
        ax.set_title(f"pred: {class_names[preds[i]]}\ntrue: {class_names[labels[i]]}",
                     color="green" if right else "red", fontsize=10)
        ax.axis("off")
    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## Run the Untrained Network
The weights are completely random right now. The model has learned nothing. With 10 classes the model will score around 10%.

In [ ]:
print(f"Untrained test accuracy: {test_accuracy(model):.1f}%   (random guessing)")
show_predictions(model, "Untrained CNN predictions (mostly wrong)")

## Train the CNN
### How training works

Training means repeatedly showing the network images, checking how wrong it is, and
nudging its weights to be a little less wrong. Two pieces set that up:

- **Loss function (`CrossEntropyLoss`):** measures how wrong the predictions are.
  Lower is better and driving this number down *is* learning.
- **Optimizer (`Adam`):** decides how to adjust the weights to reduce that loss. Its
  `lr` (learning rate) is the step size, or how big a nudge to take each time.

We loop over the whole training set `EPOCHS` times (one **epoch** = one full pass
through every image).
We loop over the whole training set `EPOCHS` number of times. One **epoch** = one full
pass through every image in the training set. This is normally done in batches.
Batches are splits of the data where you run $n$ images through the network before
computing the loss. This prevents memory errors and lets us train on larger datasets.

To update the weights, backpropagation needs the intermediate outputs the network
produced on the forward pass — and it has to keep those in memory for every image in
the batch. The more images you push through at once, the more memory that takes, so
running the entire dataset through in a single shot would blow past what the GPU can
hold. Batching keeps that memory footprint manageable. Batching also stabilizes
learning: if you updated the weights after every single image, the loss would bounce
around and the network would struggle to learn — averaging over a batch smooths each
update out.

For each batch, the same four steps run:

1. **`optimizer.zero_grad()`** — clear the previous step's gradients so they don't
   pile up.
2. **`outputs = model(images)`** — the *forward pass*: run the images through the
   network to get predictions.
3. **`loss.backward()`** — *backpropagation*: figure out how much each weight
   contributed to the error.
4. **`optimizer.step()`** — nudge every weight a small amount in the direction that
   reduces the loss.

Watch the printout as it runs: the loss should fall and the training accuracy should
climb epoch over epoch. That's the network learning in real time.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
train_losses, train_accs = [], []

for epoch in range(EPOCHS):
  model.train()
  running_loss = correct = total = 0
  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad() # clear the previous gradients
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward() # backpropagation
    optimizer.step() # gradient descent update
    running_loss += loss.item() * labels.size(0)
    correct += (outputs.argmax(1) == labels).sum().item()
    total += labels.size(0)
  train_losses.append(running_loss / total)
  train_accs.append(100 *correct / total)
  print(f"Epoch {epoch+1:2d}/{EPOCHS} loss {train_losses[-1]:.3f}   train acc {train_accs[-1]:.1f}%")

## Test the Trained Network

In [ ]:
print(f"Trained test accuracy: {test_accuracy(model):.1f}%")

## Visualize what the CNN Learned

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, train_losses, 'o-', color='steelblue', linewidth=2)
axes[0].set_title('Training Loss per Epoch', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')

axes[1].plot(epochs_range, train_accs, 'o-', color='mediumpurple', linewidth=2)
axes[1].set_title('Training Accuracy per Epoch', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')

plt.tight_layout()
plt.show()

In [ ]:
show_predictions(model, "Trained CNN predictions (no random guessing)")

## Where does the model get confused?

A single accuracy number hides *which* mistakes the model makes. A **confusion matrix**
shows, for every true class (rows), what the model actually predicted (columns). The
diagonal is correct predictions; bright off-diagonal cells are the model's blind spots.

Look for pairs that get mixed up: cats and dogs, deer and horses, automobiles and
trucks. Those confusions usually make intuitive sense: at 32×32, those classes really
do look alike.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

# Run the trained model over the whole test set, collecting predictions vs. truth
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(device)).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)
all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix (test set)", fontweight="bold")
thresh = cm.max() / 2
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black", fontsize=8)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## When the model is confidently wrong

The network doesn't just guess a class, it outputs a **confidence** for each one (via
a softmax that turns its raw scores into percentages). Usually high confidence means it's
right. But not always.

Below we hunt for the model's most **confidently wrong** predictions: images it got
wrong while being almost certain it was right. These are the failures that matter most
in the real world a model that's wrong *and* knows it's unsure can flag for review, but
one that's wrong while 99% confident will sail right past you. Worth keeping in mind
before trusting any model's output at face value.

In [ ]:
import torch.nn.functional as F

# Collect every misclassified test image along with the model's confidence in its wrong answer
model.eval()
wrong = []   # (confidence, image, predicted_label, true_label)
with torch.no_grad():
    for images, labels in test_loader:
        probs = F.softmax(model(images.to(device)), dim=1).cpu()
        conf, preds = probs.max(1)
        for i in range(len(labels)):
            if preds[i] != labels[i]:
                wrong.append((conf[i].item(), images[i], preds[i].item(), labels[i].item()))

# Sort by confidence, highest first -> the most "confidently wrong" cases
wrong.sort(key=lambda x: x[0], reverse=True)

fig, axes = plt.subplots(1, 5, figsize=(14, 3.5))
for ax, (conf, img, pred, true) in zip(axes, wrong[:5]):
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(f"says: {class_names[pred]} ({conf*100:.0f}%)\nreally: {class_names[true]}",
                 color="red", fontsize=10)
    ax.axis("off")
plt.suptitle("Confidently wrong — high confidence, wrong answer", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## What did the first layer actually learn?

We never told the network *what* to look for — we only gave it images and labels and let
training shape the filters. So what did `conv1` teach itself? Each of its 32 filters is a
tiny 3×3×3 patch of weights, which we can view directly as a little color image.

You won't see anything as obvious as "a cat," but look for structure: patches that fade
from one color to another (color detectors) or from light to dark across an edge (edge
detectors). The network invented these low-level pattern detectors on its own, the same
kinds of features the earliest layers of huge production CNNs learn too.

In [ ]:
# conv1 weights have shape (32 filters, 3 channels, 3, 3) -> view each filter as a 3x3 color image
filters = model.conv1.weight.data.clone().cpu()

# Normalize to 0-1 per filter so they're viewable as images
f_min, f_max = filters.amin(dim=(1,2,3), keepdim=True), filters.amax(dim=(1,2,3), keepdim=True)
filters = (filters - f_min) / (f_max - f_min + 1e-8)

fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i].permute(1, 2, 0))   # (3,3,3) -> (3,3,3) H,W,C
    ax.axis("off")
plt.suptitle("The 32 learned filters of conv1", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## What the network "sees"

The filters above are what the network *looks for*; the **feature maps** are what it
*finds* when it applies those filters to an actual image. We'll push one test image
through `conv1` and show the 32 resulting maps.

Each map highlights where its filter fired: bright regions are spots that matched that
filter's pattern (an edge, a color, a texture). Different maps light up on different
parts of the image: some trace outlines, some fill in flat color regions. This is the
raw material `conv2` and the dense layers build on to reach a final answer.

In [ ]:
# Grab one test image and run it through conv1 only
image, label = test_data[0]
with torch.no_grad():
    maps = model.conv1(image.unsqueeze(0).to(device))   # (1, 32, 30, 30)
maps = maps.squeeze(0).cpu()

# Show the original, then the 32 activation maps
fig = plt.figure(figsize=(11, 6))
ax0 = fig.add_subplot(4, 9, 1)
ax0.imshow(image.permute(1, 2, 0))
ax0.set_title(f"input:\n{class_names[label]}", fontsize=9); ax0.axis("off")

for i in range(32):
    ax = fig.add_subplot(4, 9, i + 2)      # slots 2..33, leaving the first for the input
    ax.imshow(maps[i], cmap="viridis")
    ax.axis("off")

plt.suptitle("conv1 feature maps — where each filter fired", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## Your turn: does more training always help?

You'd think more epochs = a better model, but there's a catch. Let's retrain a fresh
model for *longer*, and this time measure **test** accuracy (on images it never trained
on) after every epoch, alongside training accuracy.

Watch the two lines in the plot. When training accuracy keeps climbing but test
accuracy flattens out or starts to dip, the model has stopped *learning* and started
*memorizing* the training images — that gap is called **overfitting**. It's one of the
central problems in machine learning, and it's why we always judge a model on data it
hasn't seen.

**Experiment:** run the cell as-is, then try bumping `MORE_EPOCHS` higher (or changing
the learning rate `lr`) and re-run to see how the gap grows.

In [ ]:
# Fresh model so we're not continuing the one we already trained
criterion = nn.CrossEntropyLoss()          # (redefined so this cell stands alone)
exp_model = TinyCNN().to(device)
exp_optimizer = torch.optim.Adam(exp_model.parameters(), lr=1e-3)

MORE_EPOCHS = 20          # <-- try changing this and re-running
train_hist, test_hist = [], []

for epoch in range(MORE_EPOCHS):
    exp_model.train()
    correct = total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        exp_optimizer.zero_grad()
        outputs = exp_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        exp_optimizer.step()
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += labels.size(0)
    train_hist.append(100 * correct / total)
    test_hist.append(test_accuracy(exp_model))          # test pass each epoch
    print(f"Epoch {epoch+1:2d}/{MORE_EPOCHS}   train {train_hist[-1]:.1f}%   test {test_hist[-1]:.1f}%")

plt.figure(figsize=(7, 5))
epochs_range = range(1, MORE_EPOCHS + 1)
plt.plot(epochs_range, train_hist, 'o-', label='Training accuracy')
plt.plot(epochs_range, test_hist,  'o-', label='Test accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)')
plt.title('Training vs Test Accuracy — mind the gap', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
gap = train_hist[-1] - test_hist[-1]
print(f"Final: {train_hist[-1]:.1f}% on training data vs {test_hist[-1]:.1f}% on unseen test data.\n "
      f"A {gap:.1f}-point gap. That gap is overfitting: the model learned the training\n "
      f"images better than it learned the general task.")

## From this tiny CNN to real-world models

You just built, trained, and inspected a working CNN and the same ideas scale all the
way up:

- **Real architectures are this, stacked deeper.** Production models like ResNet or
  EfficientNet use the same convolve $\rightarrow$ activate $\rightarrow$ pool pattern, just with far more layers
  and clever connections between them. Nothing you saw today becomes obsolete, it becomes
  a building block.
- **You rarely start from scratch.** With **transfer learning**, you take a model already
  trained on millions of images and fine-tune it on your own (often small) dataset. You get
  strong results with a fraction of the data and compute. This is how most real image
  projects actually get built.
- **Bigger models need care, not just scale.** The overfitting gap you saw only widens with
  capacity, so real training leans on more data, data augmentation, and regularization to
  keep the model honest on data it hasn't seen.

**The takeaways worth keeping:** convolutions exploit the fact that location matters;
networks learn their own features rather than being told what to look for; and a model's
confidence is not the same as being correct. Always judge it on data it never trained on.